# Federated Analytics

This notebook serves as a base example to how to explore on datasets by calculating some different analytics functions, without directly initializing a model and a training. We will see how the mean is calculated on nodes which contain the ADNI Dataset.

### Setting the node up
Before running this notebook we configure 2 nodes: <br/>
* **Node 1 :**
  * Add a dataset `fedbiomed node -p my-first-node dataset add`
  * Select option, choose the name, tags and description of the dataset
  * Check that your data has been added by executing `fedbiomed node -p my-first-node list`
  * Run the node using `fedbiomed node -p my-first-node start`. <br/>

* **Node 2 :**
  * Repeat the same steps for `my-second-node`

Wait until you get `Starting task manager`, it means node is online.

### Setting up the experiment

We set up an experiment, in order to select the nodes which contain the datasets and perform federated analytics on them.

## Tabular Dataset

For a tabular dataset you can input which columns you want to compute the analytics on. The columns are given as a list, which can be a list of column names or column indexes.

In [1]:
from fedbiomed.researcher.federated_workflows import Experiment

# The `Experiment` scans all connected nodes for datasets matching the `'tabular'` tag. 
# Two nodes are discovered and registered for the experiment.
tags = ['tabular']
exp = Experiment(tags=tags)

2026-04-28 16:34:10,237 fedbiomed INFO - Syslog configuration is disabled or not found in configuration. Disabling syslog logging.

2026-04-28 16:34:10,341 fedbiomed INFO - Starting researcher service...

2026-04-28 16:34:10,342 fedbiomed INFO - Waiting 3s for nodes to connect...

2026-04-28 16:34:13,348 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-04-28 16:34:13,366 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_e1d11980-12dd-4fde-a356-6554d68c593d

2026-04-28 16:34:13,366 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_d0e82145-2311-46cb-93f8-922f36f4b71d

In [2]:
# Both nodes host CSV datasets with the same 7 columns and compatible types.
# Node 1 has 10 668 rows, Node 2 has 17 965 rows. 
# This metadata is returned by nodes without exposing any raw data.
exp.training_data().data()

{'NODE_e1d11980-12dd-4fde-a356-6554d68c593d': {'name': 'csv',
  'data_type': 'csv',
  'tags': ['tabular'],
  'description': 'tabular dataset',
  'shape': {'csv': [10668, 7]},
  'dtypes': {'year': 'Int64',
   'price': 'Int64',
   'transmission': 'Int64',
   'mileage': 'Int64',
   'tax': 'Int64',
   'mpg': 'Float64',
   'engineSize': 'Float64'},
  'dataset_id': 'dataset_0ca1dd3b-6bb5-41a4-86ef-2f6b3a905ff3',
  'dataset_parameters': {}},
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': {'name': 'cars',
  'data_type': 'csv',
  'tags': ['tabular'],
  'description': 'toy-dataaset',
  'shape': {'csv': [17965, 7]},
  'dtypes': {'year': 'Int64',
   'price': 'Int64',
   'transmission': 'Int64',
   'mileage': 'Int64',
   'tax': 'Int64',
   'mpg': 'Float64',
   'engineSize': 'Float64'},
  'dataset_id': 'dataset_15f67e62-e534-49e0-a080-d42849878e3d',
  'dataset_parameters': {}}}

In [3]:
# `fetch_stats('mean')` sends an analytics request to both nodes. 
# Each node computes the mean and count for every numeric column locally and returns them.
# Results are stored in an `FAResult` object and cached.
result = exp.analytics.fetch_stats("mean")

In [4]:
result.node_ids

['NODE_e1d11980-12dd-4fde-a356-6554d68c593d',
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d']

In [5]:
result.schema

{'year': {},
 'price': {},
 'transmission': {},
 'mileage': {},
 'tax': {},
 'mpg': {},
 'engineSize': {}}

In [6]:
result.node_stats()

{'NODE_e1d11980-12dd-4fde-a356-6554d68c593d': {'year': {'mean': 2017.1265,
   'count': 10667},
  'price': {'mean': 22896.543, 'count': 10668},
  'transmission': {'mean': 1.082771, 'count': 10668},
  'mileage': {'mean': 24827.29, 'count': 10668},
  'tax': {'mean': 126.01144, 'count': 10668},
  'mpg': {'mean': 50.770645, 'count': 10668},
  'engineSize': {'mean': 1.9307177, 'count': 10668}},
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': {'year': {'mean': 2016.8644,
   'count': 17965},
  'price': {'mean': 12279.554, 'count': 17965},
  'transmission': {'mean': 0.9847481, 'count': 17965},
  'mileage': {'mean': 23363.611, 'count': 17965},
  'tax': {'mean': 113.33454, 'count': 17965},
  'mpg': {'mean': 57.905792, 'count': 17965},
  'engineSize': {'mean': 1.3507986, 'count': 17965}}}

In [7]:
# `available_stats()` lists statistics present in the node replies (`count`, `mean`).
# `computable_stats()` additionally includes `sum`, which can be derived locally from `mean × count` without re-querying nodes.
print(result.available_stats())
print(result.computable_stats())

['count', 'mean']
['count', 'mean', 'sum']


In [8]:
# `global_stats('mean')` combines per-node means into a single count-weighted average across all nodes.
result.global_stats("mean")

{'year': 2016.9620209933992,
 'price': 16235.200740670456,
 'transmission': 1.0212691821941424,
 'mileage': 23908.94412840134,
 'tax': 118.05766210348528,
 'mpg': 55.24739984969232,
 'engineSize': 1.5668631812010942}

In [9]:
# `exp.analytics.mean()` is a shorthand that calls `fetch_stats('mean')` then `global_stats('mean')`. 
# The log confirms the result is served from cache — nodes are not contacted again.
exp.analytics.mean()

2026-04-28 16:34:16,679 fedbiomed INFO - All requested statistics ['mean'] are already cached — skipping node requests.

{'year': 2016.9620209933992,
 'price': 16235.200740670456,
 'transmission': 1.0212691821941424,
 'mileage': 23908.94412840134,
 'tax': 118.05766210348528,
 'mpg': 55.24739984969232,
 'engineSize': 1.5668631812010942}

In [10]:
# `min` and `max` are not recognized statistics. 
# `fetch_stats` raises a `FedbiomedError` immediately, before contacting any node. 
# The previously fetched `result` object is unaffected and still holds the mean data.
result = exp.analytics.fetch_stats(["min", "max"])

FedbiomedError: The following are not valid statistics: ['min', 'max'].

In [11]:
result.node_stats()

{'NODE_e1d11980-12dd-4fde-a356-6554d68c593d': {'year': {'mean': 2017.1265,
   'count': 10667},
  'price': {'mean': 22896.543, 'count': 10668},
  'transmission': {'mean': 1.082771, 'count': 10668},
  'mileage': {'mean': 24827.29, 'count': 10668},
  'tax': {'mean': 126.01144, 'count': 10668},
  'mpg': {'mean': 50.770645, 'count': 10668},
  'engineSize': {'mean': 1.9307177, 'count': 10668}},
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': {'year': {'mean': 2016.8644,
   'count': 17965},
  'price': {'mean': 12279.554, 'count': 17965},
  'transmission': {'mean': 0.9847481, 'count': 17965},
  'mileage': {'mean': 23363.611, 'count': 17965},
  'tax': {'mean': 113.33454, 'count': 17965},
  'mpg': {'mean': 57.905792, 'count': 17965},
  'engineSize': {'mean': 1.3507986, 'count': 17965}}}

In [12]:
# `'histogram'` passes client-side validation but is rejected by both nodes, which do not yet implement it.
# The error is fatal — no partial results are saved and the request does not overwrite the cached result.
result = exp.analytics.fetch_stats("histogram")

2026-04-28 16:34:20,972 fedbiomed ERROR - Node NODE_e1d11980-12dd-4fde-a356-6554d68c593d analytics error [FB325: Node federated analytics error]: Error during execution of analytics '['histogram']' on node='NODE_e1d11980-12dd-4fde-a356-6554d68c593d': FedbiomedError("Statistic 'histogram' is not yet supported. Only 'count', 'mean', and 'variance' are currently enabled.")

2026-04-28 16:34:20,973 fedbiomed ERROR - Node NODE_d0e82145-2311-46cb-93f8-922f36f4b71d analytics error [FB325: Node federated analytics error]: Error during execution of analytics '['histogram']' on node='NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': FedbiomedError("Statistic 'histogram' is not yet supported. Only 'count', 'mean', and 'variance' are currently enabled.")

FedbiomedError: FA request execution failed with errors from nodes: NODE_e1d11980-12dd-4fde-a356-6554d68c593d, NODE_d0e82145-2311-46cb-93f8-922f36f4b71d

In [13]:
result = exp.analytics.fetch_stats_with_args(stats_args={
    "price": {
        "histogram": {
            "bin_edges": [100,125,150,175,200,250]
        }
    }
})

2026-04-28 16:34:21,970 fedbiomed ERROR - Node NODE_e1d11980-12dd-4fde-a356-6554d68c593d analytics error [FB325: Node federated analytics error]: Error during execution of analytics 'None' on node='NODE_e1d11980-12dd-4fde-a356-6554d68c593d': FedbiomedError("Statistic 'histogram' is not yet supported. Only 'count', 'mean', and 'variance' are currently enabled.")

2026-04-28 16:34:21,971 fedbiomed ERROR - Node NODE_d0e82145-2311-46cb-93f8-922f36f4b71d analytics error [FB325: Node federated analytics error]: Error during execution of analytics 'None' on node='NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': FedbiomedError("Statistic 'histogram' is not yet supported. Only 'count', 'mean', and 'variance' are currently enabled.")

FedbiomedError: FA request execution failed with errors from nodes: NODE_e1d11980-12dd-4fde-a356-6554d68c593d, NODE_d0e82145-2311-46cb-93f8-922f36f4b71d

In [14]:
# After two failed histogram requests, `result` still holds the mean data from the original successful call.
# Failed requests never overwrite the cached result.
result.node_stats()

{'NODE_e1d11980-12dd-4fde-a356-6554d68c593d': {'year': {'mean': 2017.1265,
   'count': 10667},
  'price': {'mean': 22896.543, 'count': 10668},
  'transmission': {'mean': 1.082771, 'count': 10668},
  'mileage': {'mean': 24827.29, 'count': 10668},
  'tax': {'mean': 126.01144, 'count': 10668},
  'mpg': {'mean': 50.770645, 'count': 10668},
  'engineSize': {'mean': 1.9307177, 'count': 10668}},
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': {'year': {'mean': 2016.8644,
   'count': 17965},
  'price': {'mean': 12279.554, 'count': 17965},
  'transmission': {'mean': 0.9847481, 'count': 17965},
  'mileage': {'mean': 23363.611, 'count': 17965},
  'tax': {'mean': 113.33454, 'count': 17965},
  'mpg': {'mean': 57.905792, 'count': 17965},
  'engineSize': {'mean': 1.3507986, 'count': 17965}}}

## Mnist Dataset

In [15]:
from fedbiomed.researcher.federated_workflows import Experiment

tags = ['#MNIST', '#dataset']
exp = Experiment(tags=tags)
exp.training_data().data()

2026-04-28 16:34:24,536 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-04-28 16:34:24,547 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_e1d11980-12dd-4fde-a356-6554d68c593d

2026-04-28 16:34:24,548 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_d0e82145-2311-46cb-93f8-922f36f4b71d

{'NODE_e1d11980-12dd-4fde-a356-6554d68c593d': {'name': 'MNIST',
  'data_type': 'default',
  'tags': ['#MNIST', '#dataset'],
  'description': 'MNIST database',
  'shape': {'data': {'size': [28, 28], 'mode': 'L'}, 'target': 1},
  'dtypes': {'data': 'Image', 'target': 'int'},
  'dataset_id': 'dataset_7831794c-f690-49be-be3e-9843c810e873',
  'dataset_parameters': {'train': True, 'download': False}},
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': {'name': 'MNIST',
  'data_type': 'default',
  'tags': ['#MNIST', '#dataset'],
  'description': 'MNIST database',
  'shape': {'data': {'size': [28, 28], 'mode': 'L'}, 'target': 1},
  'dtypes': {'data': 'Image', 'target': 'int'},
  'dataset_id': 'dataset_2a8c8329-6723-4a97-b597-c632523d5a5b',
  'dataset_parameters': {'train': True, 'download': False}}}

In [16]:
result = exp.analytics.fetch_stats("mean")

In [17]:
result.node_ids

['NODE_e1d11980-12dd-4fde-a356-6554d68c593d',
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d']

In [18]:
# The schema is empty because MNIST has no named tabular columns. 
# With no columns to operate on, `available_stats`, `computable_stats`, and `node_stats` are all empty. 
# No error is raised — the request simply returns nothing.
print(result.schema)
print(result.available_stats())
print(result.computable_stats())

{}
[]
[]


In [19]:
result.node_stats()

{'NODE_e1d11980-12dd-4fde-a356-6554d68c593d': {},
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': {}}

## Medical Folder Dataset

The IXI dataset combines NIfTI image modalities (`T1`, `T2`, `label`) with a demographics CSV. Analytics can only be computed on named tabular columns; image modalities appear in the schema but produce no statistics.

In [20]:
from fedbiomed.researcher.federated_workflows import Experiment

tags = ['ixi']
exp = Experiment(tags=tags)
exp.training_data().data()

2026-04-28 16:34:30,408 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-04-28 16:34:30,418 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_e1d11980-12dd-4fde-a356-6554d68c593d

2026-04-28 16:34:30,418 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_d0e82145-2311-46cb-93f8-922f36f4b71d

{'NODE_e1d11980-12dd-4fde-a356-6554d68c593d': {'name': 'ixi',
  'data_type': 'medical-folder',
  'tags': ['ixi'],
  'description': 'ixi',
  'shape': {'label': [83, 44, 55],
   'T2': [83, 44, 55],
   'T1': [83, 44, 55],
   'demographics': 14},
  'dtypes': {'label': 'Nifti1Image',
   'T2': 'Nifti1Image',
   'T1': 'Nifti1Image',
   'demographics': 'dict'},
  'dataset_id': 'dataset_1ecd98dd-acbd-4add-8ddd-aea5d2351933',
  'dataset_parameters': {'index_col': 14}},
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': {'name': 'ixi',
  'data_type': 'medical-folder',
  'tags': ['ixi'],
  'description': 'ixi',
  'shape': {'label': [83, 44, 55],
   'T2': [83, 44, 55],
   'T1': [83, 44, 55],
   'demographics': 14},
  'dtypes': {'label': 'Nifti1Image',
   'T2': 'Nifti1Image',
   'T1': 'Nifti1Image',
   'demographics': 'dict'},
  'dataset_id': 'dataset_a4c2e5f0-b724-4a6d-bfaa-3249e116e6c9',
  'dataset_parameters': {'index_col': 14}}}

In [21]:
result = exp.analytics.fetch_stats("mean")

In [22]:
result.node_ids

['NODE_e1d11980-12dd-4fde-a356-6554d68c593d',
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d']

In [23]:
# The schema reflects the nested structure: `demographics` has named sub-columns (e.g. `AGE`, `HEIGHT`),
# while `T1`, `T2`, and `label` are image modalities with no sub-columns.
result.schema

{'demographics': {'': {},
  'IXI_ID': {},
  'SEX_ID (1=m, 2=f)': {},
  'HEIGHT': {},
  'WEIGHT': {},
  'ETHNIC_ID': {},
  'MARITAL_ID': {},
  'OCCUPATION_ID': {},
  'QUALIFICATION_ID': {},
  'DOB': {},
  'DATE_AVAILABLE': {},
  'STUDY_DATE': {},
  'AGE': {},
  'SITE_NAME': {}},
 'T1': {},
 'T2': {},
 'label': {}}

In [24]:
result.available_stats()

['count', 'mean']

In [25]:
result.node_stats()

{'NODE_e1d11980-12dd-4fde-a356-6554d68c593d': {'demographics': {'': {'mean': 265.11594,
    'count': 276},
   'IXI_ID': {'mean': 310.471, 'count': 276},
   'SEX_ID (1=m, 2=f)': {'mean': 1.5688406, 'count': 276},
   'HEIGHT': {'mean': 157.5616, 'count': 276},
   'WEIGHT': {'mean': 71.31884, 'count': 276},
   'ETHNIC_ID': {'mean': 1.3333334, 'count': 276},
   'MARITAL_ID': {'mean': 2.0434783, 'count': 276},
   'OCCUPATION_ID': {'mean': 2.4891305, 'count': 276},
   'QUALIFICATION_ID': {'mean': 3.786232, 'count': 276},
   'DOB': {'mean': nan, 'count': 0},
   'DATE_AVAILABLE': {'mean': 1.0, 'count': 276},
   'STUDY_DATE': {'mean': nan, 'count': 0},
   'AGE': {'mean': 51.366047, 'count': 275},
   'SITE_NAME': {'mean': nan, 'count': 0}},
  'T1': {},
  'T2': {},
  'label': {}},
 'NODE_d0e82145-2311-46cb-93f8-922f36f4b71d': {'demographics': {'': {'mean': 284.7044,
    'count': 159},
   'IXI_ID': {'mean': 331.8239, 'count': 159},
   'SEX_ID (1=m, 2=f)': {'mean': 1.509434, 'count': 159},
   'HEIG

Only numeric demographics columns have valid stats. `DOB`, `STUDY_DATE`, and `SITE_NAME` return `nan` with `count: 0` because they are non-numeric (dates or text). Image modality keys (`T1`, `T2`, `label`) are empty — per-voxel statistics are not computed.